# Extended Sensitivity Validation

Stress-tests the composite index: rebuilds it from raw data and verifies it, runs Monte Carlo (Dirichlet) and Sobol-type global sensitivity on the four main weights, compares the severity weights with AHP, PCA and equal weights, tests robust (median/MAD) normalization and bootstraps a confidence band per canton.

Outputs: sensitivity tables in `data/processed/`.

## 0. Setup

In [ ]:
# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helper to load the YAML config
from src.utilidades.config_utils import load_config
# Project logger factory
from src.utilidades.logging_utils import get_logger

# Numerical operations
import numpy as np
# Tabular data handling
import pandas as pd
# Spearman rank correlation
from scipy.stats import spearmanr
# Principal component analysis
from sklearn.decomposition import PCA
# Natural-breaks (Jenks) classification
import jenkspy

# Show up to 40 columns when printing DataFrames
pd.set_option("display.max_columns", 40)

In [ ]:
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)
# Folder with processed tables
PROCESSED = PROJECT_ROOT / "data" / "processed"

# Create a logger that writes to the logs folder
logger = get_logger("step13_sensitivity_validation", config, log_dir=PROJECT_ROOT / "logs")
# Log that the notebook started and which config was used
logger.info("sensibilidad iniciada — configuración cargada desde %s", CONFIG_PATH)

# Severity index settings from the config
SEV_CFG = config["severity_index"]
# Weight multiplier for destroyed assets versus affected ones
W_DESTROYED = SEV_CFG["destroyed_weight_multiplier"]
# Weight multiplier for deaths and missing persons
W_DEATH = SEV_CFG["death_weight_multiplier"]
# Weight multiplier for injured persons
W_INJURED = SEV_CFG["injured_weight_multiplier"]
# Base weights of the five severity pillars
PILLAR_WEIGHTS_BASE = SEV_CFG["pillar_weights"]
# Names of the five pillars
PILLARS = ["poblacion", "vivienda", "agricultura", "ganaderia", "infraestructura"]

# Composite index settings from the config
EFCI_CFG = config["efci_composite"]
# Base weights of the four composite-index components
EFCI_WEIGHTS_BASE = EFCI_CFG["weights"]
# Names of the four components
COMPONENTS = ["recurrencia", "severidad", "persistencia", "magnitud_enso"]
# Number of classes
N_CLASSES = EFCI_CFG["n_classes"]
# Class labels, from lowest to highest
CLASS_LABELS = EFCI_CFG["class_labels"]

# Random generator with a fixed seed for reproducibility
rng = np.random.default_rng(20260904)
# Log the base weights
logger.info("Pesos EFCI base: %s | Pesos Severidad base: %s", EFCI_WEIGHTS_BASE, PILLAR_WEIGHTS_BASE)

## 1. Rebuild from raw data and verify

In [ ]:
# Load the event table
he = pd.read_csv(PROCESSED / "hazard_event.csv")
# Log the number of events
logger.info("hazard_event.csv: %d filas", len(he))

# Population score: affected, evacuated and sheltered people plus weighted deaths, missing and injured
he["score_poblacion"] = (
    he["affected_direct"] + he["affected_indirect"] + he["evacuated"] + he["sheltered"]
    + W_DEATH * (he["deaths"] + he["missing_persons"])
    + W_INJURED * he["injured"]
)
# Housing score: affected houses plus weighted destroyed houses
he["score_vivienda"] = he["houses_affected"] + W_DESTROYED * he["houses_destroyed"]
# Agriculture score: affected hectares plus weighted destroyed hectares
he["score_agricultura"] = he["crops_affected_ha"] + W_DESTROYED * he["crops_destroyed_ha"]
# Livestock score: affected animals plus weighted deaths
he["score_ganaderia"] = he["livestock_affected"] + W_DESTROYED * he["livestock_deaths"]
# Infrastructure score: affected and destroyed schools and health facilities
he["score_infraestructura"] = (
    he["schools_affected"] + W_DESTROYED * he["schools_destroyed"]
    + he["health_facilities_affected"] + W_DESTROYED * he["health_facilities_destroyed"]
)
# Names of the pillar score columns
SCORE_COLS = [f"score_{p}" for p in PILLARS]
# Format canton codes as 4-digit strings
he["canton_code"] = he["canton_code"].apply(lambda c: f"{int(c):04d}")
# Confirm the pillar scores were recomputed
print("OK: puntajes de pilar recalculados a nivel evento")

In [ ]:
# Group events by canton
g = he.groupby("canton_code")
# Maximum of each pillar per canton
sev_max = g[SCORE_COLS].max().reset_index()
# Number of events per canton
sev_max["n_eventos"] = g.size().values

# Min-max normalization of the selected columns
def minmax(df, cols):
    # Work on a copy
    df = df.copy()
    # Loop over the columns
    for c in cols:
        # Name of the normalized column
        norm_c = c.replace("score_", "norm_")
        # Range of the column
        rango = df[c].max() - df[c].min()
        # Min-max scaling, zero for a constant column
        df[norm_c] = 0.0 if rango == 0 else (df[c] - df[c].min()) / rango
    # Return the table with the index
    return df

# Normalize the pillar maxima
sev_norm = minmax(sev_max, SCORE_COLS)
# Names of the normalized columns
NORM_COLS = [f"norm_{p}" for p in PILLARS]

# Severity as a weighted sum of the normalized pillars
def severity_from_weights(df, weights: dict):
    # Return the weighted sum
    return sum(weights[p] * df[f"norm_{p}"] for p in PILLARS)

# Recompute the severity index with the base weights
sev_norm["severity_index_recalc"] = severity_from_weights(sev_norm, PILLAR_WEIGHTS_BASE)

# Load the official severity index
sev_oficial = pd.read_csv(PROCESSED / "severity_index_canton.csv")
# Keep the combined hazard view
sev_oficial_todos = sev_oficial[sev_oficial["hazard_type"] == "TODOS"].copy()
# Format canton codes as 4-digit strings
sev_oficial_todos["canton_code"] = sev_oficial_todos["canton_code"].apply(lambda c: f"{int(c):04d}")

# Join the recomputed and the official values
chk = sev_norm.merge(sev_oficial_todos[["canton_code", "severity_index"]], on="canton_code", how="inner")
# Absolute difference between both
diff = (chk["severity_index_recalc"] - chk["severity_index"]).abs()
# Print the number of rows compared and the largest difference
print(f"Filas comparadas: {len(chk)} | Diferencia máxima: {diff.max():.2e}")
# Check the rebuilt severity equals the official one
assert diff.max() < 1e-9, "La reconstrucción de severity_index no coincide con severidad"
# Confirm the rebuilt severity matches exactly
print("OK: severity_index reconstruido coincide EXACTAMENTE con severity_index_canton.csv")

In [ ]:
# Load the combined hazard view of one index table
def cargar_todos(nombre, col_code="canton_code"):
    # Read the table
    df = pd.read_csv(PROCESSED / nombre)
    # Keep the combined hazard view
    df = df[df["hazard_type"] == "TODOS"].copy()
    # Format canton codes as 4-digit strings
    df[col_code] = df[col_code].apply(lambda c: f"{int(c):04d}")
    # Return the table with the index
    return df

# Recurrence index
rec = cargar_todos("recurrence_index_canton.csv")[["canton_code", "recurrence_index"]]
# Persistence index
per = cargar_todos("persistence_index_canton.csv")[["canton_code", "persistence_index"]]
# ENSO magnitude (mean and maximum)
mag = cargar_todos("enso_magnitude_index_canton.csv")[["canton_code", "enso_magnitude_mean", "enso_magnitude_max"]]

# Base table with severity and normalized pillars
base = sev_norm[["canton_code", "n_eventos", "severity_index_recalc"] + NORM_COLS].copy()
# Rename the recomputed severity
base = base.rename(columns={"severity_index_recalc": "severity_index"})
# Join recurrence, persistence and magnitude
base = base.merge(rec, on="canton_code", how="left").merge(per, on="canton_code", how="left").merge(mag, on="canton_code", how="left")
# Check there are no missing values after the joins
assert base.isna().sum().sum() == 0, "hay nulos tras unir los 4 índices — revisar cobertura"
# Confirm full coverage after the joins
print("OK: cobertura completa tras unir Recurrencia/Persistencia/Magnitud (0 nulos)")

# Composite index from a set of weights
def calcular_efci(df, w_efci: dict, severity_col="severity_index", magnitude_col="enso_magnitude_mean"):
    # Weighted sum of the four components
    return (
        w_efci["recurrencia"] * df["recurrence_index"]
        + w_efci["severidad"] * df[severity_col]
        + w_efci["persistencia"] * df["persistence_index"]
        + w_efci["magnitud_enso"] * df[magnitude_col]
    )

# Recompute the composite index with the base weights
base["efci_recalc"] = calcular_efci(base, EFCI_WEIGHTS_BASE)

# Load the official composite index
efci_oficial = pd.read_csv(PROCESSED / "efci_composite_canton.csv")
# Keep the combined hazard view
efci_oficial_todos = efci_oficial[efci_oficial["hazard_type"] == "TODOS"].copy()
# Format canton codes as 4-digit strings
efci_oficial_todos["canton_code"] = efci_oficial_todos["canton_code"].apply(lambda c: f"{int(c):04d}")

# Join the recomputed and the official values
chk2 = base.merge(efci_oficial_todos[["canton_code", "efci", "clase_efci"]], on="canton_code", how="inner")
# Absolute difference between both
diff2 = (chk2["efci_recalc"] - chk2["efci"]).abs()
# Print the number of rows compared and the largest difference
print(f"Filas comparadas (EFCI): {len(chk2)} | Diferencia máxima: {diff2.max():.2e}")
# Check the rebuilt index equals the official one
assert diff2.max() < 1e-9, "La reconstrucción de efci no coincide con índice compuesto"
# Confirm the rebuilt index matches exactly
print("OK: efci reconstruido coincide EXACTAMENTE con efci_composite_canton.csv")

In [ ]:
# Jenks natural-breaks classification
def clasificar_jenks(serie: pd.Series, n_classes: int, labels: list) -> pd.Series:
    """Classify values with Jenks natural breaks.
    
    Args:
        serie: Values to classify.
        n_classes: Number of classes.
        labels: Class labels, from lowest to highest.
    
    Returns:
        The class of each value.
    """
    # Compute the class boundaries
    breaks = list(jenkspy.jenks_breaks(serie.values, n_classes=n_classes))
    # Widen the lowest boundary so the minimum value is included
    breaks[0] -= 1e-9
    # Assign each value to its class
    return pd.cut(serie, bins=breaks, labels=labels, include_lowest=True)

# Table used by all sensitivity tests
BASE = chk2.copy()
# Reclassify the recomputed index
BASE["clase_efci_recalc"] = clasificar_jenks(BASE["efci_recalc"], N_CLASSES, CLASS_LABELS)
# Check the rebuilt classes equal the official ones
assert (BASE["clase_efci_recalc"].astype(str) == BASE["clase_efci"].astype(str)).all(), \
    "clasificación Jenks reconstruida no coincide con índice compuesto"
# Confirm the rebuilt classification matches exactly
print("OK: clasificación Jenks reconstruida coincide EXACTAMENTE con índice compuesto")
# Print that the rebuild is complete
print("\nReconstrucción completa y verificada — se procede con las pruebas de sensibilidad.")

## 2. Monte Carlo (Dirichlet) and Sobol-type global sensitivity on the four index weights

In [ ]:
# Number of Monte Carlo samples
N_MC = 10000
# Base weights as a vector
base_vec = np.array([EFCI_WEIGHTS_BASE[c] for c in COMPONENTS])

# Draw random weight vectors from a Dirichlet distribution
def muestrear_dirichlet(base_vec, concentracion, n, rng):
    # Concentration parameters, with a floor to avoid zeros
    alpha = np.clip(base_vec * concentracion, 0.05, None)
    # Draw the samples
    return rng.dirichlet(alpha, size=n)

# Concentration that keeps weights close to the base values
CONC_CENTRADA = 420
# Weights sampled around the base values
muestras_centrada = muestrear_dirichlet(base_vec, CONC_CENTRADA, N_MC, rng)
# Check which samples stay within +-20% of the base weights
dentro_20pct = np.all(np.abs(muestras_centrada - base_vec) <= 0.20 * base_vec, axis=1)
# Print the share of samples within +-20%
print(f"Calibración corrida centrada: {dentro_20pct.mean()*100:.1f}% de las muestras dentro de +-20% de todos los pesos base")

# Uninformative samples: uniform Dirichlet over the whole simplex
muestras_uniforme = rng.dirichlet(np.ones(4), size=N_MC)

# Evaluate each weight sample against the base index
def evaluar_muestras(muestras, base_df, clase_base_col="clase_efci"):
    # List that collects one row per sample
    resultados = []
    # Base index values
    y_efci_base = base_df["efci_recalc"].values
    # Base classes
    clase_base = base_df[clase_base_col].astype(str).values
    # Number of cantons
    n = len(base_df)
    # Loop over the weight samples
    for w in muestras:
        # Weights as a dictionary
        w_dict = dict(zip(COMPONENTS, w))
        # Composite index with the sampled weights
        efci_mc = calcular_efci(base_df, w_dict)
        # Spearman correlation with the base index
        rho, _ = spearmanr(y_efci_base, efci_mc)
        # Classes with the sampled weights
        clase_mc = clasificar_jenks(pd.Series(efci_mc, index=base_df.index), N_CLASSES, CLASS_LABELS).astype(str).values
        # Percentage of cantons that change class
        pct_cambio = 100 * (clase_mc != clase_base).sum() / n
        # Store the weights, correlation and class change
        resultados.append((*w, rho, pct_cambio))
    # Return the results as a table
    return pd.DataFrame(resultados, columns=COMPONENTS + ["spearman_rho", "pct_cambio_clase"])

# Evaluate the centered samples
res_centrada = evaluar_muestras(muestras_centrada, BASE)
# Evaluate the uniform samples
res_uniforme = evaluar_muestras(muestras_uniforme, BASE)

# Print a header for the centered Monte Carlo
print("\n=== Monte Carlo centrado en pesos base (N=%d) ===" % N_MC)
# Print summary statistics
print(res_centrada[["spearman_rho", "pct_cambio_clase"]].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))
# Print a header for the uninformative Monte Carlo
print("\n=== Monte Carlo no informativo (Dirichlet uniforme, N=%d) ===" % N_MC)
# Print summary statistics
print(res_uniforme[["spearman_rho", "pct_cambio_clase"]].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

# Save the centered results
res_centrada.to_csv(PROCESSED / "mc_efci_pesos_centrada.csv", index=False)
# Save the uniform results
res_uniforme.to_csv(PROCESSED / "mc_efci_pesos_uniforme.csv", index=False)
# Log the export
logger.info("Monte Carlo exportado: mc_efci_pesos_centrada.csv, mc_efci_pesos_uniforme.csv")

### 2.1 Which weight moves the result most? (given-data estimator)

In [ ]:
# Sobol-type first-order index estimated from the given samples
def sobol_given_data(df_muestras, y_col, weight_cols, n_bins=20):
    # Output values
    y = df_muestras[y_col].values
    # Total variance of the output
    var_total = y.var()
    # List that collects one row per unit
    filas = []
    # Loop over the weights
    for w in weight_cols:
        # Split the weight into equal-count bins
        bins = pd.qcut(df_muestras[w], q=n_bins, duplicates="drop")
        # Mean of the output per bin
        medias_bin = df_muestras.groupby(bins, observed=True)[y_col].mean()
        # Number of samples per bin
        conteos_bin = df_muestras.groupby(bins, observed=True)[y_col].size()
        # Global mean of the output
        media_global = np.average(medias_bin, weights=conteos_bin)
        # Variance between bin means
        var_entre_bins = np.average((medias_bin - media_global) ** 2, weights=conteos_bin)
        # Share of the total variance explained by the weight
        s1 = var_entre_bins / var_total if var_total > 0 else np.nan
        # Store the weight and its index
        filas.append({"peso": w, "S1_aprox": round(s1, 4)})
    # Return the indices sorted from largest to smallest
    return pd.DataFrame(filas).sort_values("S1_aprox", ascending=False).reset_index(drop=True)

# Print a header for the sensitivity results
print("=== Sensibilidad global tipo-Sobol (S1 aproximado, Y=% cambio de clase) ===")
# Print the centered-sampling label
print("-- Muestreo centrado (+-20% del peso base) --")
# Compute the indices for the centered samples
sobol_centrada = sobol_given_data(res_centrada, "pct_cambio_clase", COMPONENTS)
# Print the indices and their sum
print(sobol_centrada, f"\n(suma S1 = {sobol_centrada.S1_aprox.sum():.3f} — no tiene por qué sumar 1 exacto: los pesos no son independientes, la diferencia son interacciones/efectos de orden superior)")

# Print the uniform-sampling label
print("\n-- Muestreo no informativo (simplex completo) --")
# Compute the indices for the uniform samples
sobol_uniforme = sobol_given_data(res_uniforme, "pct_cambio_clase", COMPONENTS)
# Print the indices and their sum
print(sobol_uniforme, f"\n(suma S1 = {sobol_uniforme.S1_aprox.sum():.3f})")

# Save the centered indices
sobol_centrada.to_csv(PROCESSED / "sobol_s1_efci_pesos_centrada.csv", index=False)
# Save the uniform indices
sobol_uniforme.to_csv(PROCESSED / "sobol_s1_efci_pesos_uniforme.csv", index=False)
# Log the export
logger.info("Sobol S1 exportado: sobol_s1_efci_pesos_centrada.csv, sobol_s1_efci_pesos_uniforme.csv")

## 3. AHP, PCA and equal weights for the five severity weights

In [ ]:
# Convert a weight ratio to Saaty's 1-9 scale
def a_escala_saaty(ratio):
    # Ratios of at least 1
    if ratio >= 1:
        # Round and cap at 9
        return float(np.clip(round(ratio), 1, 9))
    # Inverse of the capped ratio for ratios below 1
    return 1.0 / float(np.clip(round(1 / ratio), 1, 9))

# Base severity weights as a vector
w_base_vec = np.array([PILLAR_WEIGHTS_BASE[p] for p in PILLARS])
# Number of pillars
n_p = len(PILLARS)
# Start the pairwise comparison matrix with ones
M_ahp = np.ones((n_p, n_p))
# Loop over rows
for i in range(n_p):
    # Loop over columns
    for j in range(n_p):
        # Only off-diagonal entries
        if i != j:
            # Pairwise comparison derived from the weight ratio
            M_ahp[i, j] = a_escala_saaty(w_base_vec[i] / w_base_vec[j])

# Print a header for the matrix
print("Matriz de comparación pareada AHP (Saaty), derivada de los pesos base:")
# Print the AHP matrix
print(pd.DataFrame(M_ahp, index=PILLARS, columns=PILLARS).round(2))

# Eigenvalues and eigenvectors of the matrix
eigvals, eigvecs = np.linalg.eig(M_ahp)
# Position of the largest eigenvalue
idx_max = np.argmax(eigvals.real)
# Largest eigenvalue
lambda_max = eigvals.real[idx_max]
# Principal eigenvector
w_ahp = np.abs(eigvecs[:, idx_max].real)
# Normalize it to add up to 1
w_ahp = w_ahp / w_ahp.sum()

# Consistency index
CI = (lambda_max - n_p) / (n_p - 1)
# Random index table for the consistency ratio
RI_TABLE = {1: 0.0, 2: 0.0, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}
# Consistency ratio
CR = CI / RI_TABLE[n_p]

# Print the AHP weights
print(f"\nPesos AHP (autovector principal): {dict(zip(PILLARS, w_ahp.round(4)))}")
# Print the largest eigenvalue, CI and CR
print(f"lambda_max={lambda_max:.4f}  CI={CI:.4f}  CR={CR:.4f} "
      f"({'aceptable, CR<0.10' if CR < 0.10 else 'INACEPTABLE, CR>=0.10 -- matriz inconsistente'})")

# AHP weights as a dictionary
W_AHP = dict(zip(PILLARS, w_ahp))

In [ ]:
# Normalized pillar matrix
X = sev_norm[NORM_COLS].values
# PCA with as many components as pillars
pca = PCA(n_components=n_p)
# Fit the PCA
pca.fit(X)
# Absolute loadings of the first component
cargas_pc1 = np.abs(pca.components_[0])
# PCA weights: loadings normalized to add up to 1
w_pca = cargas_pc1 / cargas_pc1.sum()
# Variance explained by the first component
var_explicada_pc1 = pca.explained_variance_ratio_[0]
# Print the variance explained
print(f"Pesos PCA (cargas |PC1|, normalizadas a sumar 1) — PC1 explica {var_explicada_pc1*100:.1f}% de la varianza:")
# Print the PCA weights
print(dict(zip(PILLARS, w_pca.round(4))))
# PCA weights as a dictionary
W_PCA = dict(zip(PILLARS, w_pca))

# Equal weights
W_EQUAL = {p: 1 / n_p for p in PILLARS}

# Alternative severity weight schemes
ESQUEMAS_SEVERIDAD = {"ahp": W_AHP, "pca": W_PCA, "equitativo": W_EQUAL}
# Table comparing all schemes
tabla_pesos = pd.DataFrame({"base": PILLAR_WEIGHTS_BASE, **ESQUEMAS_SEVERIDAD})
# Print a header for the comparison
print("\n=== Comparación de esquemas de pesos de Severidad ===")
# Print the comparison
print(tabla_pesos.round(4))

In [ ]:
# List that collects the results of each scheme
resultados_severidad = []
# Table of class results per canton
tabla_variantes_severidad = BASE[["canton_code", "clase_efci"]].copy()

# Loop over the schemes
for nombre, pesos in ESQUEMAS_SEVERIDAD.items():
    # Severity with the scheme's weights
    sev_alt = severity_from_weights(sev_norm, pesos)
    # Copy of the base table
    df_alt = BASE.copy()
    # Add the alternative severity
    df_alt["severity_index_alt"] = sev_alt.values
    # Composite index with the alternative severity
    efci_alt = calcular_efci(df_alt, EFCI_WEIGHTS_BASE, severity_col="severity_index_alt")
    # Classes of the alternative index
    clase_alt = clasificar_jenks(efci_alt, N_CLASSES, CLASS_LABELS).astype(str)
    # Spearman correlation with the base index
    rho, p = spearmanr(BASE["efci_recalc"], efci_alt)
    # Number of cantons that change class
    cambios = (clase_alt.values != BASE["clase_efci"].astype(str).values).sum()
    # Percentage that change class
    pct = 100 * cambios / len(BASE)
    # Store the results of the scheme
    resultados_severidad.append({
        "esquema": nombre, "spearman_rho": round(rho, 4),
        "cantones_cambian_clase": int(cambios), "pct_cambian_clase": round(pct, 1),
    })
    # Add the alternative index to the canton table
    tabla_variantes_severidad[f"efci_sev_{nombre}"] = efci_alt.values
    # Add the alternative classes
    tabla_variantes_severidad[f"clase_efci_sev_{nombre}"] = clase_alt.values
    # Print the correlation and class changes
    print(f"Severidad-{nombre:12s} rho={rho:.4f}  cantones que cambian de clase: {cambios}/{len(BASE)} ({pct:.1f}%)")

# Results table
tabla_resultados_severidad = pd.DataFrame(resultados_severidad)
# Save the comparison
tabla_resultados_severidad.to_csv(PROCESSED / "severity_weights_comparison.csv", index=False)
# Save the per-canton comparison
tabla_variantes_severidad.to_csv(PROCESSED / "severity_weights_comparison_canton.csv", index=False)
# Save the weight schemes
tabla_pesos.to_csv(PROCESSED / "severity_weights_schemes.csv")
# Save the AHP consistency measures
pd.DataFrame([{"lambda_max": lambda_max, "CI": CI, "CR": CR, "n_pilares": n_p}]).to_csv(PROCESSED / "ahp_consistency.csv", index=False)
# Log the export and the consistency ratio
logger.info("Comparación de esquemas de Severidad exportada (AHP CR=%.4f)", CR)

## 4. Robust normalization (median/MAD)

In [ ]:
# Robust normalization using the median and the median absolute deviation (MAD)
def normalizar_robusta_mad(df, cols, clip=3.5):
    # Work on a copy
    df = df.copy()
    # Loop over the columns
    for c in cols:
        # Name of the robust column
        norm_c = c.replace("score_", "normrobusta_")
        # Median of the column
        mediana = df[c].median()
        # Median absolute deviation
        mad = (df[c] - mediana).abs().median()
        # Robust standard deviation
        sigma_robusto = 1.4826 * mad
        # Robust z-score, zero if there is no dispersion
        z = (df[c] - mediana) / sigma_robusto if sigma_robusto > 0 else pd.Series(0.0, index=df.index)
        # Clip extreme z-scores
        z_clip = z.clip(-clip, clip)
        # Range after clipping
        rango = z_clip.max() - z_clip.min()
        # Scale to 0-1
        df[norm_c] = 0.0 if rango == 0 else (z_clip - z_clip.min()) / rango
    # Return the table with the index
    return df

# Apply the robust normalization to the pillar maxima
sev_norm_robusta = normalizar_robusta_mad(sev_max, SCORE_COLS)
# Severity index from the robust pillars
sev_norm_robusta["severity_index_mad"] = sum(
    PILLAR_WEIGHTS_BASE[p] * sev_norm_robusta[f"normrobusta_{p}"] for p in PILLARS
)

# Copy of the base table
df_mad = BASE.copy()
# Add the robust severity
df_mad["severity_index_mad"] = sev_norm_robusta["severity_index_mad"].values
# Composite index with the robust severity
efci_mad = calcular_efci(df_mad, EFCI_WEIGHTS_BASE, severity_col="severity_index_mad")
# Classes of the robust index
clase_mad = clasificar_jenks(efci_mad, N_CLASSES, CLASS_LABELS).astype(str)
# Spearman correlation with the base index
rho_mad, _ = spearmanr(BASE["efci_recalc"], efci_mad)
# Number of cantons that change class
cambios_mad = (clase_mad.values != BASE["clase_efci"].astype(str).values).sum()
# Percentage that change class
pct_mad = 100 * cambios_mad / len(BASE)
# Print the correlation and class changes
print(f"Severidad-mediana/MAD   rho={rho_mad:.4f}  cantones que cambian de clase: {cambios_mad}/{len(BASE)} ({pct_mad:.1f}%)")
# Print the reference result from the winsorization test
print("(Referencia índice compuesto, winsorización p99: rho=0.8716, 131/221 cantones, 59.3%)")

# Save the robust-normalization summary
pd.DataFrame([{
    "metodo": "mediana_mad", "spearman_rho": round(rho_mad, 4),
    "cantones_cambian_clase": int(cambios_mad), "pct_cambian_clase": round(pct_mad, 1),
}]).to_csv(PROCESSED / "severity_robust_mad_comparison.csv", index=False)

# Canton names lookup
canton_lookup = efci_oficial_todos[["canton_code", "canton_name_official"]].drop_duplicates()
# Attach names to the robust table
df_mad_named = df_mad.merge(canton_lookup, on="canton_code", how="left")
# Print an empty line
print()
# Loop over the three reference cantons
for nombre in ["Chone", "Distrito Metropolitano de Quito", "Baños de Agua Santa"]:
    # Select the canton row
    row = df_mad_named[df_mad_named["canton_name_official"].str.contains(nombre, case=False, na=False)]
    # Only if the canton was found
    if len(row):
        # First matching row
        r = row.iloc[0]
        # Print the base and robust severity of the canton
        print(f"  {nombre}: severity_index base={r['severity_index']:.3f}  mediana/MAD={r['severity_index_mad']:.3f}")

## 5. Bootstrap uncertainty: confidence band per canton

In [ ]:
# Number of bootstrap resamples
N_BOOT = 1000
# Random generator for the bootstrap
rng_boot = np.random.default_rng(20260904 + 1)

# List of cantons
cantones = sev_max["canton_code"].values
# Pillar scores of the events of each canton
score_matrix_por_canton = {cc: he.loc[he["canton_code"] == cc, SCORE_COLS].values for cc in cantones}

# Array that stores the bootstrap maxima
boot_max = np.zeros((N_BOOT, len(cantones), len(SCORE_COLS)))
# Loop over the cantons
for i, cc in enumerate(cantones):
    # Event scores of the canton
    vals = score_matrix_por_canton[cc]
    # Number of events of the canton
    n = vals.shape[0]
    # Resample the events with replacement
    idx = rng_boot.integers(0, n, size=(N_BOOT, n))
    # Maximum of each pillar in every resample
    boot_max[:, i, :] = vals[idx].max(axis=1)

# Min-max normalization across cantons for every resample
def minmax_matrix(mat):
    # Minimum per resample
    mn = mat.min(axis=1, keepdims=True)
    # Maximum per resample
    mx = mat.max(axis=1, keepdims=True)
    # Range per resample
    rango = mx - mn
    # Avoid division by zero
    rango[rango == 0] = 1.0
    # Return the normalized array
    return (mat - mn) / rango

# Normalize the bootstrap maxima
boot_norm = minmax_matrix(boot_max)
# Severity weights as a vector
w_vec_pillars = np.array([PILLAR_WEIGHTS_BASE[p] for p in PILLARS])
# Bootstrap severity index
boot_severity = boot_norm @ w_vec_pillars

# Recurrence of each canton
rec_vec = BASE.set_index("canton_code").loc[cantones, "recurrence_index"].values
# Persistence of each canton
per_vec = BASE.set_index("canton_code").loc[cantones, "persistence_index"].values
# ENSO magnitude of each canton
mag_vec = BASE.set_index("canton_code").loc[cantones, "enso_magnitude_mean"].values

# Bootstrap composite index
boot_efci = (
    EFCI_WEIGHTS_BASE["recurrencia"] * rec_vec[None, :]
    + EFCI_WEIGHTS_BASE["severidad"] * boot_severity
    + EFCI_WEIGHTS_BASE["persistencia"] * per_vec[None, :]
    + EFCI_WEIGHTS_BASE["magnitud_enso"] * mag_vec[None, :]
)

# 5th, 50th and 95th percentiles of the composite index
p5, p50, p95 = np.percentile(boot_efci, [5, 50, 95], axis=0)
# Percentiles of the severity index
p5_sev, p50_sev, p95_sev = np.percentile(boot_severity, [5, 50, 95], axis=0)

# Table with the uncertainty bands
tabla_boot = pd.DataFrame({
    "canton_code": cantones,
    "efci_base": BASE.set_index("canton_code").loc[cantones, "efci_recalc"].values,
    "efci_p5": p5, "efci_p50": p50, "efci_p95": p95,
    "severity_index_base": BASE.set_index("canton_code").loc[cantones, "severity_index"].values,
    "severity_p5": p5_sev, "severity_p50": p50_sev, "severity_p95": p95_sev,
    "n_eventos": sev_max["n_eventos"].values,
})
# Width of the 90% interval
tabla_boot["efci_ic90_ancho"] = tabla_boot["efci_p95"] - tabla_boot["efci_p5"]
# Attach canton names
tabla_boot = tabla_boot.merge(canton_lookup, on="canton_code", how="left")
# Save the bootstrap table
tabla_boot.to_csv(PROCESSED / "bootstrap_ci_efci_canton.csv", index=False)

# Print a header for the interval widths
print(f"=== Bootstrap EFCI (N={N_BOOT}) — resumen del ancho de banda IC90 ===")
# Print summary statistics of the widths
print(tabla_boot["efci_ic90_ancho"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

In [ ]:
# Print a header for the widest bands
print("Top 5 cantones con banda de incertidumbre MÁS ANCHA:")
# Print the five cantons with the widest bands
print(tabla_boot.nlargest(5, "efci_ic90_ancho")[
    ["canton_name_official", "n_eventos", "efci_base", "efci_p5", "efci_p95", "efci_ic90_ancho"]].to_string(index=False))

# Print a header for the narrowest bands
print("\nTop 5 cantones con banda de incertidumbre MÁS ANGOSTA:")
# Print the five cantons with the narrowest bands
print(tabla_boot.nsmallest(5, "efci_ic90_ancho")[
    ["canton_name_official", "n_eventos", "efci_base", "efci_p5", "efci_p95", "efci_ic90_ancho"]].to_string(index=False))

# Print a header for the reference cantons
print("\nCasos de referencia (etapas previas):")
# Print the bands of the reference cantons
print(tabla_boot[tabla_boot["canton_name_official"].isin(
    ["Chone", "Distrito Metropolitano de Quito", "Baños de Agua Santa"])][
    ["canton_name_official", "n_eventos", "efci_base", "efci_p5", "efci_p50", "efci_p95"]].to_string(index=False))

## 6. Consolidated summary

In [ ]:
# Consolidated table of all tests
resumen_final = pd.DataFrame([
    {"prueba": "Monte Carlo EFCI (centrado +-20%, N=10000) - mediana pct cambio clase", "valor": res_centrada["pct_cambio_clase"].median()},
    {"prueba": "Monte Carlo EFCI (centrado +-20%, N=10000) - p95 pct cambio clase", "valor": res_centrada["pct_cambio_clase"].quantile(0.95)},
    {"prueba": "Monte Carlo EFCI (simplex completo, N=10000) - mediana pct cambio clase", "valor": res_uniforme["pct_cambio_clase"].median()},
    {"prueba": "Sobol S1 aprox (centrado) - peso mas influyente: magnitud_enso", "valor": sobol_centrada.iloc[0]["S1_aprox"]},
    {"prueba": "Sobol S1 aprox (simplex completo) - peso mas influyente: magnitud_enso", "valor": sobol_uniforme.iloc[0]["S1_aprox"]},
    {"prueba": "AHP Severidad - Consistency Ratio", "valor": CR},
    {"prueba": "Severidad-AHP vs base - pct cambian clase", "valor": tabla_resultados_severidad.loc[tabla_resultados_severidad.esquema=="ahp","pct_cambian_clase"].iloc[0]},
    {"prueba": "Severidad-PCA vs base - pct cambian clase", "valor": tabla_resultados_severidad.loc[tabla_resultados_severidad.esquema=="pca","pct_cambian_clase"].iloc[0]},
    {"prueba": "Severidad-equitativo vs base - pct cambian clase", "valor": tabla_resultados_severidad.loc[tabla_resultados_severidad.esquema=="equitativo","pct_cambian_clase"].iloc[0]},
    {"prueba": "Severidad-mediana/MAD vs base - pct cambian clase", "valor": pct_mad},
    {"prueba": "Bootstrap EFCI - ancho IC90 mediano (todos los cantones)", "valor": tabla_boot["efci_ic90_ancho"].median()},
    {"prueba": "Bootstrap EFCI - ancho IC90 p95 (todos los cantones)", "valor": tabla_boot["efci_ic90_ancho"].quantile(0.95)},
])
# Round the values
resumen_final["valor"] = resumen_final["valor"].round(4)
# Print the consolidated summary
print(resumen_final.to_string(index=False))
# Save the consolidated summary
resumen_final.to_csv(PROCESSED / "sensibilidad_resumen.csv", index=False)
# Log that all tests were completed
logger.info("sensibilidad completa — %d pruebas consolidadas en sensibilidad_resumen.csv", len(resumen_final))